# 02 — Extracción Tabular e Inspección Exploratoria de Postulaciones
**Proyecto:** Youth Employment Bridge (YEB) — Módulo de Analítica de Empleo  
**Issue:** `#04 [Data/Inspection]: Extracción tabular e inspección exploratoria de postulaciones`  
**Criterios de Aceptación:** `CA-ISS-04-01`, `CA-ISS-04-02`, `CA-ISS-04-03`, `CA-ISS-04-04`  
**Riesgos mitigados:** `OPE-005`, `TEC-010` | **Casos de prueba:** `CP-044`, `CP-044B`

---
### Objetivos del Notebook
1. Leer las extracciones anonimizadas de `jobs` y `applications` desde `data/raw/` empleando **rutas relativas (`pathlib.Path`)**.
2. Ejecutar y documentar la **inspección técnica estructural** mediante `shape`, `columns`, `dtypes`, `info()` y `describe()`.
3. Calcular **métricas salariales y tasas porcentuales de avance a entrevista** utilizando **operaciones vectorizadas en NumPy** (sin bucles iterativos manuales).

## 1. Configuración de Rutas Relativas (`pathlib.Path`) y Carga desde `data/raw/` (`CA-ISS-04-01`, `CA-ISS-04-02`)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Resolución dinámica de rutas relativas (sin rutas absolutas fijas)
cwd = Path.cwd()
if cwd.name == "notebooks":
    ANALYTICS_ROOT = cwd.parent
elif (cwd / "analytics").is_dir():
    ANALYTICS_ROOT = cwd / "analytics"
else:
    ANALYTICS_ROOT = cwd

RAW_DIR = ANALYTICS_ROOT / "data" / "raw"
JOBS_CSV = RAW_DIR / "yeb_jobs_raw.csv"
APPS_CSV = RAW_DIR / "yeb_applications_raw.csv"
EXCEL_DUMP = RAW_DIR / "yeb_extraccion_tabular.xlsx"

assert JOBS_CSV.exists(), f"No se encontró {JOBS_CSV}"
assert APPS_CSV.exists(), f"No se encontró {APPS_CSV}"
assert EXCEL_DUMP.exists(), f"No se encontró {EXCEL_DUMP}"

df_jobs = pd.read_csv(JOBS_CSV)
df_apps = pd.read_csv(APPS_CSV)

print(f"[OK] Vacantes cargadas desde      : analytics/{JOBS_CSV.relative_to(ANALYTICS_ROOT).as_posix()}")
print(f"[OK] Postulaciones cargadas desde : analytics/{APPS_CSV.relative_to(ANALYTICS_ROOT).as_posix()}")
print(f"[OK] Libro Excel disponible en    : analytics/{EXCEL_DUMP.relative_to(ANALYTICS_ROOT).as_posix()}")

[OK] Vacantes cargadas desde      : analytics/data/raw/yeb_jobs_raw.csv
[OK] Postulaciones cargadas desde : analytics/data/raw/yeb_applications_raw.csv
[OK] Libro Excel disponible en    : analytics/data/raw/yeb_extraccion_tabular.xlsx


## 2. Inspección Técnica de la Tabla `jobs` (`shape`, `columns`, `dtypes`, `info()`, `describe()`) (`CA-ISS-04-03`)
Se audita la estructura tabular de las ofertas laborales para prevenir el riesgo **`TEC-010`** (inconsistencias de tipos en ingesta).

In [2]:
print("=== 2.1 Dimensiones (shape) y Columnas (columns) de `jobs` ===")
print("shape  :", df_jobs.shape)
print("columns:", df_jobs.columns.tolist())

print("\n=== 2.2 Tipos de datos (dtypes) de `jobs` ===")
display(df_jobs.dtypes.to_frame(name="dtype"))

print("\n=== 2.3 Resumen estructural info() de `jobs` ===")
df_jobs.info()

print("\n=== 2.4 Estadísticos descriptivos describe() de `jobs` ===")
display(df_jobs.describe(include="all").T)

=== 2.1 Dimensiones (shape) y Columnas (columns) de `jobs` ===
shape  : (20, 12)
columns: ['job_id', 'company_id', 'company_name', 'industry', 'title', 'city', 'work_mode', 'experience', 'salary_min', 'salary_max', 'status', 'created_at']

=== 2.2 Tipos de datos (dtypes) de `jobs` ===


,dtype
job_id,int64
company_id,int64
company_name,str
industry,str
title,str
city,str
work_mode,str
experience,str
salary_min,float64
salary_max,float64



=== 2.3 Resumen estructural info() de `jobs` ===
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   job_id        20 non-null     int64  
 1   company_id    20 non-null     int64  
 2   company_name  20 non-null     str    
 3   industry      20 non-null     str    
 4   title         20 non-null     str    
 5   city          20 non-null     str    
 6   work_mode     20 non-null     str    
 7   experience    20 non-null     str    
 8   salary_min    20 non-null     float64
 9   salary_max    20 non-null     float64
 10  status        20 non-null     str    
 11  created_at    20 non-null     str    
dtypes: float64(2), int64(2), str(8)
memory usage: 2.0 KB

=== 2.4 Estadísticos descriptivos describe() de `jobs` ===


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
job_id,20.0,NaN,NaN,NaN,10.5,5.91608,1.0,5.75,10.5,15.25,20.0
company_id,20.0,NaN,NaN,NaN,4.2,2.307881,1.0,2.0,4.0,6.0,8.0
company_name,20,8,Talento Lab,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
industry,20,7,Tecnología,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,20,20,Frontend Developer Junior,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,20,6,Bogotá,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
work_mode,20,3,Hybrid,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
experience,20,3,Inicial,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
salary_min,20.0,NaN,NaN,NaN,1862500.0,601286.996892,1350000.0,1437500.0,1600000.0,2175000.0,3200000.0
salary_max,20.0,NaN,NaN,NaN,2417500.0,912237.705633,1550000.0,1737500.0,2075000.0,2825000.0,4500000.0


## 3. Inspección Técnica de la Tabla `applications` (`shape`, `columns`, `dtypes`, `info()`, `describe()`) (`CA-ISS-04-03`)
Se verifica que la extracción de postulaciones cumpla con la política de anonimización (`candidate_hash`, sin nombres ni cédulas en claro) y que las edades estén dentro del rango juvenil de YEB (`16` a `24` años).

In [3]:
print("=== 3.1 Dimensiones (shape) y Columnas (columns) de `applications` ===")
print("shape  :", df_apps.shape)
print("columns:", df_apps.columns.tolist())

print("\n=== 3.2 Tipos de datos (dtypes) de `applications` ===")
display(df_apps.dtypes.to_frame(name="dtype"))

print("\n=== 3.3 Resumen estructural info() de `applications` ===")
df_apps.info()

print("\n=== 3.4 Estadísticos descriptivos describe() de `applications` ===")
display(df_apps.describe(include="all").T)

print("\n=== 3.5 Muestra de los primeros 5 registros de `applications` ===")
display(df_apps.head())

=== 3.1 Dimensiones (shape) y Columnas (columns) de `applications` ===
shape  : (120, 12)
columns: ['application_id', 'job_id', 'user_id', 'candidate_hash', 'candidate_age', 'is_minor', 'minor_authorized', 'candidate_city', 'expected_salary', 'status', 'created_at', 'updated_at']

=== 3.2 Tipos de datos (dtypes) de `applications` ===


,dtype
application_id,int64
job_id,int64
user_id,int64
candidate_hash,str
candidate_age,int64
is_minor,int64
minor_authorized,int64
candidate_city,str
expected_salary,float64
status,str



=== 3.3 Resumen estructural info() de `applications` ===
<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   application_id    120 non-null    int64  
 1   job_id            120 non-null    int64  
 2   user_id           120 non-null    int64  
 3   candidate_hash    120 non-null    str    
 4   candidate_age     120 non-null    int64  
 5   is_minor          120 non-null    int64  
 6   minor_authorized  120 non-null    int64  
 7   candidate_city    120 non-null    str    
 8   expected_salary   120 non-null    float64
 9   status            120 non-null    str    
 10  created_at        120 non-null    str    
 11  updated_at        120 non-null    str    
dtypes: float64(1), int64(6), str(5)
memory usage: 11.4 KB

=== 3.4 Estadísticos descriptivos describe() de `applications` ===


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
application_id,120.0,NaN,NaN,NaN,60.5,34.785054,1.0,30.75,60.5,90.25,120.0
job_id,120.0,NaN,NaN,NaN,9.5,5.365624,1.0,5.0,10.0,14.0,19.0
user_id,120.0,NaN,NaN,NaN,134.0,18.71188,101.0,118.75,134.0,150.25,164.0
candidate_hash,120,52,CAND-92e580badb7f,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
candidate_age,120.0,NaN,NaN,NaN,20.225,2.720796,16.0,18.0,20.0,23.0,24.0
is_minor,120.0,NaN,NaN,NaN,0.208333,0.407819,0.0,0.0,0.0,0.0,1.0
minor_authorized,120.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
candidate_city,120,5,Cali,27,NaN,NaN,NaN,NaN,NaN,NaN,NaN
expected_salary,120.0,NaN,NaN,NaN,2018583.333333,672248.573266,1300000.0,1557500.0,1770000.0,2510000.0,3620000.0
status,120,5,review,36,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== 3.5 Muestra de los primeros 5 registros de `applications` ===


,application_id,job_id,user_id,candidate_hash,candidate_age,is_minor,minor_authorized,candidate_city,expected_salary,status,created_at,updated_at
0,1,15,106,CAND-f71954baad00,18,0,1,Medellín,1860000.0,review,2026-10-01 13:30:00,2026-10-01 15:30:00
1,2,11,107,CAND-1cbb83f80504,17,1,1,Cali,1790000.0,accepted,2026-09-25 12:00:00,2026-10-04 13:00:00
2,3,9,154,CAND-c2efa6c68961,24,0,1,Bucaramanga,1460000.0,review,2026-09-26 16:00:00,2026-09-29 19:00:00
3,4,11,153,CAND-f7354995e880,16,1,1,Barranquilla,1470000.0,pending,2026-09-14 15:00:00,2026-09-16 16:00:00
4,5,16,155,CAND-dcbceff44a8f,23,0,1,Bogotá,1550000.0,interview,2026-09-30 20:00:00,2026-10-09 21:00:00


## 4. Operaciones Vectorizadas en NumPy: Rangos Salariales Medios y Tasas de Avance a Entrevista (`CA-ISS-04-04`)
Para mitigar el riesgo **`OPE-005`**, todos los cálculos numéricos se ejecutan mediante arreglos `ndarray` de **NumPy** y operaciones vectorizadas (0 bucles `for` sobre filas):
- **Punto medio salarial por vacante**: $\text{Midpoint}_i = \frac{\text{salary\_min}_i + \text{salary\_max}_i}{2}$
- **Amplitud salarial (Rango)**: $\text{Spread}_i = \text{salary\_max}_i - \text{salary\_min}_i$
- **Tasa de avance a entrevista**: porcentaje de postulaciones que alcanzan la etapa `interview` o `accepted`.

In [4]:
# 4.1 Operaciones vectorizadas en NumPy sobre salarios ofertados (vacantes publicadas)
mask_published = (df_jobs["status"].to_numpy() == "published")
sal_min_arr = df_jobs.loc[mask_published, "salary_min"].to_numpy(dtype=np.float64)
sal_max_arr = df_jobs.loc[mask_published, "salary_max"].to_numpy(dtype=np.float64)

sal_midpoint_arr = (sal_min_arr + sal_max_arr) / 2.0
sal_spread_arr = sal_max_arr - sal_min_arr

# 4.2 Operaciones vectorizadas sobre expectativas salariales y estados en applications
expected_sal_arr = df_apps["expected_salary"].to_numpy(dtype=np.float64)
status_arr = df_apps["status"].to_numpy()
is_minor_arr = df_apps["is_minor"].to_numpy(dtype=bool)

mask_interview_current = (status_arr == "interview")
mask_interview_reached = np.isin(status_arr, ["interview", "accepted"])
mask_accepted = (status_arr == "accepted")
mask_rejected = (status_arr == "rejected")

metrics_summary = pd.DataFrame([
    {"Indicador_Vectorizado_NumPy": "Salario mínimo ofertado medio (COP)", "Valor": round(float(np.mean(sal_min_arr)), 2)},
    {"Indicador_Vectorizado_NumPy": "Salario máximo ofertado medio (COP)", "Valor": round(float(np.mean(sal_max_arr)), 2)},
    {"Indicador_Vectorizado_NumPy": "Punto medio salarial promedio de vacantes (COP)", "Valor": round(float(np.mean(sal_midpoint_arr)), 2)},
    {"Indicador_Vectorizado_NumPy": "Mediana del punto medio salarial (COP)", "Valor": round(float(np.median(sal_midpoint_arr)), 2)},
    {"Indicador_Vectorizado_NumPy": "Amplitud (rango) salarial promedio (COP)", "Valor": round(float(np.mean(sal_spread_arr)), 2)},
    {"Indicador_Vectorizado_NumPy": "Expectativa salarial media del candidato joven (COP)", "Valor": round(float(np.mean(expected_sal_arr)), 2)},
    {"Indicador_Vectorizado_NumPy": "Tasa actual en etapa 'interview' (%)", "Valor": round(float(np.mean(mask_interview_current) * 100.0), 2)},
    {"Indicador_Vectorizado_NumPy": "Tasa acumulada de avance a entrevista (interview + accepted) (%)", "Valor": round(float(np.mean(mask_interview_reached) * 100.0), 2)},
    {"Indicador_Vectorizado_NumPy": "Tasa de contratación efectiva ('accepted') (%)", "Valor": round(float(np.mean(mask_accepted) * 100.0), 2)},
    {"Indicador_Vectorizado_NumPy": "Tasa de rechazo ('rejected') (%)", "Valor": round(float(np.mean(mask_rejected) * 100.0), 2)},
    {"Indicador_Vectorizado_NumPy": "Avance a entrevista — Menores Protegidos 16-17 años (%)", "Valor": round(float(np.mean(mask_interview_reached[is_minor_arr]) * 100.0), 2)},
    {"Indicador_Vectorizado_NumPy": "Avance a entrevista — Jóvenes Adultos 18-24 años (%)", "Valor": round(float(np.mean(mask_interview_reached[~is_minor_arr]) * 100.0), 2)},
])

display(metrics_summary)

,Indicador_Vectorizado_NumPy,Valor
0,Salario mínimo ofertado medio (COP),1863888.89
1,Salario máximo ofertado medio (COP),2416666.67
2,Punto medio salarial promedio de vacantes (COP),2140277.78
3,Mediana del punto medio salarial (COP),1787500.00
4,Amplitud (rango) salarial promedio (COP),552777.78
5,Expectativa salarial media del candidato joven...,2018583.33
6,Tasa actual en etapa 'interview' (%),25.00
7,Tasa acumulada de avance a entrevista (intervi...,35.00
8,Tasa de contratación efectiva ('accepted') (%),10.00
9,Tasa de rechazo ('rejected') (%),13.33


## 5. Desglose por Modalidad de Trabajo (`Remote`, `Hybrid`, `Presencial`)

In [5]:
df_merged = df_apps.merge(
    df_jobs[["job_id", "title", "work_mode", "city", "salary_min", "salary_max"]],
    on="job_id",
    how="left"
)
df_merged["job_salary_midpoint"] = (df_merged["salary_min"].to_numpy() + df_merged["salary_max"].to_numpy()) / 2.0
df_merged["advanced_to_interview"] = np.isin(df_merged["status"].to_numpy(), ["interview", "accepted"])

by_modality = (
    df_merged.groupby("work_mode", as_index=False)
    .agg(
        total_postulaciones=("application_id", "count"),
        salario_medio_ofertado_cop=("job_salary_midpoint", "mean"),
        expectativa_media_candidato_cop=("expected_salary", "mean"),
        tasa_avance_entrevista_pct=("advanced_to_interview", lambda s: round(float(np.mean(s.to_numpy()) * 100.0), 2)),
    )
    .round(2)
)

display(by_modality)

,work_mode,total_postulaciones,salario_medio_ofertado_cop,expectativa_media_candidato_cop,tasa_avance_entrevista_pct
0,Hybrid,42,2103571.43,1956190.48,30.95
1,Presencial,47,2001595.74,1891063.83,31.91
2,Remote,31,2454838.71,2296451.61,45.16
